In [0]:
# Bad practice: wildcard imports, unused imports, unused variables
from pyspark.sql import *
from pyspark.sql.functions import *
import math, json, re  # Unused imports

unused_variable = 12345
HARDCODED_THRESHOLD = 9999
another_unused = "this will never be used"


In [0]:
# Hardcoded column names, poor aliasing, unnecessary caching
sales_df = spark.table("myfirstcatalog.myfirstproduct.sales").cache()
stores_df = spark.table("myfirstcatalog.myfirstproduct.stores").cache()

# Bad aliases and duplicated column usage
joined_df = sales_df.alias("s").join(
    stores_df.alias("st"), col("s.Store_ID") == col("st.Store_ID"), "inner"
)

# Extra useless select with duplicated columns, hardcoded string literals
dirty_df = joined_df.select(
    col("st.Store_Name"),
    col("s.Product_ID"),
    col("s.Units"),
    col("s.Store_ID"),
    col("st.Store_ID"),
    lit("HARDCODED").alias("Dummy"),
    (col("s.Units") * 1).alias("UnitsCopy")
)


In [0]:
# Redundant grouping, unused intermediate dataframe
intermediate_df = dirty_df.groupBy("Store_Name").agg(
    sum("Units").alias("TotalUnits"),
    count("Product_ID").alias("ProductSold"),
    max("UnitsCopy").alias("MaxUnit")
)

# Bad: nested aggregation over already aggregated data
final_df = intermediate_df.groupBy("Store_Name").agg(
    sum("TotalUnits").alias("GrandTotal"),
    sum("ProductSold").alias("GrandProducts"),
    max("MaxUnit").alias("WorstNamedColumn")
)


In [0]:
# Dead code: final_df is never used properly
final_result = final_df.filter(col("GrandTotal") >= 0).orderBy("GrandTotal")

# Unused show() call, collect() just for side effect
final_result.show(10, False)
final_result.collect()  # Collecting big data is a bad practice

# Another dataframe created but never used
unused_df = sales_df.groupBy("Store_ID").count()
